In [ ]:
%pip install pyarrow

In [ ]:
# ============================================
# PROYECTO: ENCUESTA DE HOGARES 2025 - BOLIVIA
# Notebook: 01_inspeccion_microdatos
# Objetivo: inspección inicial de las bases SPSS
# ============================================

from pathlib import Path
import pandas as pd
import pyreadstat
import duckdb
import json
from datetime import datetime

print("Librerías importadas correctamente.")
print("Pandas:", pd.__version__)
print("DuckDB:", duckdb.__version__)
print("Pyreadstat:", pyreadstat.__version__)

In [ ]:
# ============================================
# CONFIGURACIÓN DE RUTAS DEL PROYECTO
# ============================================

# Buscar automáticamente la raíz del proyecto
current_dir = Path.cwd()

candidates = [current_dir, *current_dir.parents]

PROJECT_DIR = next(
    (path for path in candidates
     if (path / "data" / "raw").is_dir()),
    None
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "No se encontró la carpeta data/raw. "
        "Revisa la ubicación del proyecto."
    )

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"

print("Directorio del proyecto:", PROJECT_DIR)
print("Carpeta de microdatos:", RAW_DIR)
print("¿Existe la carpeta?:", RAW_DIR.exists())

In [ ]:
# ============================================
# INVENTARIO INICIAL DE ARCHIVOS SPSS
# ============================================

archivos_sav = sorted(RAW_DIR.glob("*.sav"))

print(f"Total de archivos SPSS encontrados: {len(archivos_sav)}")
print("\nArchivos disponibles:\n")

for archivo in archivos_sav:
    tamano_mb = archivo.stat().st_size / (1024 ** 2)
    print(f"{archivo.name} - {tamano_mb:.2f} MB")

In [ ]:
# ============================================
# INVENTARIO GENERAL DE LAS BASES EH2025
# ============================================

inventario = []

for archivo in archivos_sav:

    # Leer únicamente los metadatos
    _, meta = pyreadstat.read_sav(
        str(archivo),
        metadataonly=True
    )

    inventario.append({
        "archivo": archivo.name,
        "registros": meta.number_rows,
        "variables": meta.number_columns,
        "etiqueta_base": meta.file_label
    })

# Convertir a DataFrame
df_inventario = pd.DataFrame(inventario)

# Mostrar resultados
display(df_inventario)

# Guardar inventario
TABLES_DIR.mkdir(parents=True, exist_ok=True)

df_inventario.to_csv(
    TABLES_DIR / "inventario_general.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nInventario guardado correctamente.")

In [ ]:
# ============================================
# DICCIONARIO DE VARIABLES EH2025
# ============================================

diccionario = []

for archivo in archivos_sav:

    _, meta = pyreadstat.read_sav(
        str(archivo),
        metadataonly=True
    )

    for nombre, etiqueta in zip(
        meta.column_names,
        meta.column_labels
    ):

        diccionario.append({
            "archivo": archivo.name,
            "variable": nombre,
            "etiqueta": etiqueta
        })

df_diccionario = pd.DataFrame(diccionario)

# Mostrar las primeras 20 variables
display(df_diccionario.head(20))

# Guardar diccionario completo
df_diccionario.to_csv(
    TABLES_DIR / "diccionario_variables.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nTotal de variables identificadas:", len(df_diccionario))
print("Diccionario guardado correctamente.")

In [ ]:
# ============================================
# ETIQUETAS DE VALORES SPSS
# ============================================

etiquetas_valores = []

for archivo in archivos_sav:

    _, meta = pyreadstat.read_sav(
        str(archivo),
        metadataonly=True
    )

    for variable, etiquetas in meta.variable_value_labels.items():

        for codigo, descripcion in etiquetas.items():

            etiquetas_valores.append({
                "archivo": archivo.name,
                "variable": variable,
                "codigo": codigo,
                "descripcion": descripcion
            })

df_etiquetas = pd.DataFrame(etiquetas_valores)

display(df_etiquetas.head(30))

df_etiquetas.to_csv(
    TABLES_DIR / "etiquetas_valores.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nTotal de etiquetas identificadas:", len(df_etiquetas))

In [ ]:
# ============================================
# VARIABLES DE LAS BASES PRINCIPALES
# ============================================

bases_principales = [
    "EH2025_Discriminacion.sav",
    "EH2025_Persona.sav",
    "EH2025_Seguridad_Alimentaria.sav"
]

for archivo in bases_principales:

    print("\n" + "=" * 70)
    print(f"BASE: {archivo}")
    print("=" * 70)

    variables = df_diccionario[
        df_diccionario["archivo"] == archivo
    ]

    for _, fila in variables.iterrows():
        print(f"{fila['variable']:25} | {fila['etiqueta']}")

In [ ]:
# ============================================
# IDENTIFICACIÓN DE POSIBLES LLAVES DE UNIÓN
# ============================================

import re

patron = re.compile(
    r"folio|nro|vivienda|hogar|persona|upm|estrato|depto|area",
    re.IGNORECASE
)

posibles_llaves = df_diccionario[
    df_diccionario["variable"].str.contains(
        patron,
        na=False
    )
]

display(
    posibles_llaves.sort_values(
        ["archivo", "variable"]
    ).reset_index(drop=True)
)

In [ ]:
# ============================================
# VARIABLES DE DISCRIMINACIÓN Y SUS CATEGORÍAS
# ============================================

archivo_objetivo = "EH2025_Discriminacion.sav"

variables_discriminacion = df_diccionario[
    df_diccionario["archivo"] == archivo_objetivo
]

for variable in variables_discriminacion["variable"]:

    print("\n" + "-" * 60)
    print("VARIABLE:", variable)

    categorias = df_etiquetas[
        (df_etiquetas["archivo"] == archivo_objetivo) &
        (df_etiquetas["variable"] == variable)
    ]

    if not categorias.empty:
        display(categorias[["codigo", "descripcion"]])

In [ ]:

# Comprobar las claves de identificación en Discriminación y Persona

archivo_dis = RAW_DIR / "EH2025_Discriminacion.sav"
archivo_per = RAW_DIR / "EH2025_Persona.sav"

# Leer únicamente las variables identificadoras
df_dis_ids, _ = pyreadstat.read_sav(
    str(archivo_dis),
    usecols=["folio", "nro"]
)

df_per_ids, _ = pyreadstat.read_sav(
    str(archivo_per),
    usecols=["folio", "nro"]
)

# Comprobar duplicados y valores faltantes
for nombre, df in [
    ("Discriminación", df_dis_ids),
    ("Persona", df_per_ids)
]:
    print(f"\n{'=' * 55}")
    print(f"BASE: {nombre}")
    print(f"{'=' * 55}")
    print("Total de registros:", len(df))
    print("Folios vacíos:", df["folio"].isna().sum())
    print("Números de persona vacíos:", df["nro"].isna().sum())
    print(
        "Duplicados de folio + nro:",
        df.duplicated(subset=["folio", "nro"]).sum()
    )
    print("Combinaciones únicas:", df[["folio", "nro"]].drop_duplicates().shape[0])

# Comprobar coincidencias entre ambas bases
ids_dis = df_dis_ids.drop_duplicates(subset=["folio", "nro"])
ids_per = df_per_ids.drop_duplicates(subset=["folio", "nro"])

comparacion = ids_dis.merge(
    ids_per,
    on=["folio", "nro"],
    how="outer",
    indicator=True
)

print("\n" + "=" * 55)
print("COMPARACIÓN ENTRE LAS DOS BASES")
print("=" * 55)
print(comparacion["_merge"].value_counts())


In [ ]:

# Revisar la estructura de las bases según el identificador del hogar

bases_hogar = [
    "EH2025_Equipamiento.sav",
    "EH2025_Gastos_Alimentarios.sav",
    "EH2025_Gastos_No_Alimentarios.sav",
    "EH2025_Seguridad_Alimentaria.sav",
    "EH2025_Vivienda_1.sav"
]

revision_hogares = []

for nombre in bases_hogar:
    ruta = RAW_DIR / nombre

    # Leer únicamente el identificador del hogar
    df_ids = pyreadstat.read_sav(
        str(ruta),
        usecols=["folio"]
    )[0]

    revision_hogares.append({
        "base": nombre,
        "registros": len(df_ids),
        "folios_vacios": df_ids["folio"].isna().sum(),
        "hogares_unicos": df_ids["folio"].nunique(),
        "folios_repetidos": df_ids.duplicated(
            subset=["folio"]
        ).sum(),
        "max_registros_por_hogar": (
            df_ids.groupby("folio").size().max()
        )
    })

df_revision_hogares = pd.DataFrame(revision_hogares)

display(df_revision_hogares)


In [ ]:

# Comprobar coincidencia de hogares entre las bases de nivel hogar

bases_comparar = [
    "EH2025_Vivienda_1.sav",
    "EH2025_Seguridad_Alimentaria.sav",
    "EH2025_Gastos_No_Alimentarios.sav",
    "EH2025_Equipamiento.sav",
    "EH2025_Gastos_Alimentarios.sav"
]

conjuntos_hogares = {}

for nombre in bases_comparar:
    df_ids = pyreadstat.read_sav(
        str(RAW_DIR / nombre),
        usecols=["folio"]
    )[0]

    conjuntos_hogares[nombre] = set(
        df_ids["folio"].dropna().unique()
    )

# Tomar Vivienda como referencia
hogares_referencia = conjuntos_hogares[
    "EH2025_Vivienda_1.sav"
]

for nombre, hogares in conjuntos_hogares.items():
    print("\nBASE:", nombre)
    print("Hogares únicos:", len(hogares))
    print(
        "Hogares que coinciden con Vivienda:",
        len(hogares & hogares_referencia)
    )
    print(
        "Hogares que no están en Vivienda:",
        len(hogares - hogares_referencia)
    )
    print(
        "Hogares de Vivienda que no están en esta base:",
        len(hogares_referencia - hogares)
    )


In [ ]:

# Buscar variables relevantes para nuestro análisis

import re

patron_analisis = re.compile(
    r"sexo|edad|idioma|ind[ií]gena|educaci[oó]n|"
    r"estudio|alfabet|actividad|ocupaci[oó]n|"
    r"ingreso|trabajo|parentesco|"
    r"vivienda|habitaci[oó]n|pared|techo|piso|"
    r"agua|sanitario|ba[nñ]o|alcantarillado|"
    r"electricidad|combustible|"
    r"alimento|comer|hambre|dinero|"
    r"seguridad alimentaria|factor|ponderador",
    re.IGNORECASE
)

bases_interes = [
    "EH2025_Persona.sav",
    "EH2025_Vivienda_1.sav",
    "EH2025_Seguridad_Alimentaria.sav"
]

for archivo in bases_interes:
    print("\n" + "=" * 85)
    print("BASE:", archivo)
    print("=" * 85)

    variables = df_diccionario[
        (df_diccionario["archivo"] == archivo) &
        (
            df_diccionario["variable"].str.contains(
                patron_analisis, na=False
            ) |
            df_diccionario["etiqueta"].str.contains(
                patron_analisis, na=False
            )
        )
    ]

    for _, fila in variables.iterrows():
        print(f"{fila['variable']:25} | {fila['etiqueta']}")


In [ ]:

# Revisar variables demográficas y sus categorías

variables_demograficas = [
    "s01a_02",  # Sexo (por verificar)
    "s01a_03",  # Edad (por verificar)
    "s01a_08",  # Idioma aprendido en la niñez
    "s01a_09",  # Autoidentificación indígena
    "s03a_03a", # Nivel educativo anterior
    "s04b_12"   # Categoría ocupacional
]

archivo_objetivo = "EH2025_Persona.sav"

for variable in variables_demograficas:
    print("\n" + "=" * 65)
    print("VARIABLE:", variable)
    print("=" * 65)

    # Mostrar la etiqueta de la variable
    fila = df_diccionario[
        (df_diccionario["archivo"] == archivo_objetivo) &
        (df_diccionario["variable"] == variable)
    ]

    if fila.empty:
        print("No encontrada en el diccionario.")
        continue

    print("Etiqueta:", fila.iloc[0]["etiqueta"])

    # Mostrar categorías codificadas
    categorias = df_etiquetas[
        (df_etiquetas["archivo"] == archivo_objetivo) &
        (df_etiquetas["variable"] == variable)
    ]

    if categorias.empty:
        print("No se encontraron etiquetas de valores.")
    else:
        display(categorias[["codigo", "descripcion"]])


In [ ]:

# Inspección de las variables objetivo de discriminación

archivo_objetivo = RAW_DIR / "EH2025_Discriminacion.sav"

variables_objetivo = [
    f"s09a_01{letra}" for letra in "abcdefghijkl"
] + ["s09a_02"]

# Leer únicamente las variables que vamos a estudiar
df_objetivo, meta_objetivo = pyreadstat.read_sav(
    str(archivo_objetivo),
    usecols=variables_objetivo
)

for variable in variables_objetivo:
    print("\n" + "=" * 70)
    print("VARIABLE:", variable)
    print("=" * 70)

    # Mostrar la pregunta
    etiqueta = dict(
        zip(meta_objetivo.column_names, meta_objetivo.column_labels)
    ).get(variable, "Sin etiqueta")

    print("Pregunta:", etiqueta)

    # Mostrar las categorías y sus frecuencias
    frecuencias = df_objetivo[variable].value_counts(
        dropna=False
    ).sort_index()

    etiquetas = meta_objetivo.variable_value_labels.get(
        variable, {}
    )

    for codigo, cantidad in frecuencias.items():
        if pd.isna(codigo):
            descripcion = "Valor vacío / perdido"
        else:
            descripcion = etiquetas.get(
                codigo, "Sin etiqueta de valor"
            )

        porcentaje = cantidad / len(df_objetivo) * 100

        print(
            f"Código: {codigo} | "
            f"Respuesta: {descripcion} | "
            f"Frecuencia: {cantidad:,} | "
            f"Porcentaje: {porcentaje:.2f}%"
        )


In [ ]:

# Resumen completo de los motivos de discriminación

motivos = [f"s09a_01{x}" for x in "abcdefghijkl"]

resumen_discriminacion = []

for variable in motivos:
    serie = df_objetivo[variable]

    resumen_discriminacion.append({
        "variable": variable,
        "si": (serie == 1).sum(),
        "no": (serie == 2).sum(),
        "prefiere_no_decir": (serie == 3).sum(),
        "no_sabe": (serie == 4).sum(),
        "vacios": serie.isna().sum(),
        "porcentaje_si": round(
            (serie == 1).sum() / len(serie) * 100, 2
        )
    })

df_resumen_discriminacion = pd.DataFrame(
    resumen_discriminacion
)

display(df_resumen_discriminacion)

# Guardar resultados
df_resumen_discriminacion.to_csv(
    TABLES_DIR / "resumen_motivos_discriminacion.csv",
    index=False,
    encoding="utf-8-sig"
)


In [ ]:

# Construcción del indicador general de discriminación

motivos = [f"s09a_01{x}" for x in "abcdefghijkl"]

respuestas = df_objetivo[motivos]

# Identificar las tres situaciones
tiene_discriminacion = respuestas.eq(1).any(axis=1)

todos_no = respuestas.eq(2).all(axis=1)

# Crear una nueva clasificación
df_objetivo["discriminacion_general"] = 2

df_objetivo.loc[todos_no, "discriminacion_general"] = 0

df_objetivo.loc[
    tiene_discriminacion,
    "discriminacion_general"
] = 1

# Contar cuántos motivos afirmativos reportó cada persona
df_objetivo["cantidad_motivos"] = respuestas.eq(1).sum(axis=1)

# Resumen de la clasificación
etiquetas = {
    0: "No reportó discriminación",
    1: "Reportó al menos un motivo",
    2: "Respuesta indeterminada"
}

resumen_general = (
    df_objetivo["discriminacion_general"]
    .value_counts()
    .sort_index()
    .rename(index=etiquetas)
    .to_frame("personas")
)

resumen_general["porcentaje"] = (
    resumen_general["personas"] /
    len(df_objetivo) * 100
).round(2)

print("CLASIFICACIÓN GENERAL DE DISCRIMINACIÓN")
display(resumen_general)

print("\nCANTIDAD DE MOTIVOS REPORTADOS")
display(
    df_objetivo["cantidad_motivos"]
    .value_counts()
    .sort_index()
    .to_frame("personas")
)

# Guardar la clasificación
df_objetivo.to_csv(
    TABLES_DIR / "indicador_discriminacion.csv",
    index=False,
    encoding="utf-8-sig"
)


In [ ]:
# Relación entre discriminación general y denuncia formal

tabla_denuncias = pd.crosstab(
    df_objetivo["discriminacion_general"],
    df_objetivo["s09a_02"],
    margins=True,
    dropna=False
)

display(tabla_denuncias)

In [ ]:
# Filtrar personas que reportaron al menos un motivo

personas_con_discriminacion = df_objetivo[
    df_objetivo["discriminacion_general"] == 1
]

print("Personas que reportaron discriminación:",
      len(personas_con_discriminacion))

print("\nRespuestas sobre denuncia formal:")

display(
    personas_con_discriminacion["s09a_02"]
    .value_counts(dropna=False)
    .to_frame("personas")
)

In [ ]:
# Filtrar personas que reportaron al menos un motivo
personas_con_discriminacion = df_objetivo[
    df_objetivo["discriminacion_general"] == 1
].copy()

# Resumen de denuncia formal
resumen_denuncias = (
    personas_con_discriminacion["s09a_02"]
    .value_counts()
    .rename(index={
        1: "Presentó denuncia",
        2: "No presentó denuncia"
    })
    .to_frame("personas")
)

resumen_denuncias["porcentaje"] = (
    resumen_denuncias["personas"] /
    len(personas_con_discriminacion) * 100
).round(2)

display(resumen_denuncias)

# Guardar resultado
resumen_denuncias.to_csv(
    TABLES_DIR / "resumen_denuncias.csv",
    encoding="utf-8-sig"
)

In [ ]:

# ============================================================
# PASO 1: CONSTRUCCIÓN DE LA BASE ANALÍTICA DEL CASO 3
# ============================================================

# Variables de discriminación
variables_motivos = [f"s09a_01{x}" for x in "abcdefghijkl"]

variables_discriminacion = [
    "folio", "nro", "depto", "area",
    "upm", "estrato", "factor",
    *variables_motivos,
    "s09a_02"
]

# Variables sociodemográficas
variables_persona = [
    "folio", "nro",
    "s01a_02",       # Sexo
    "s01a_03",       # Edad
    "s01a_08",       # Primer idioma aprendido
    "s01a_09",       # Autoidentificación indígena
    "s01a_09npioc",  # Nación o pueblo de autoidentificación
    "s03a_03a",      # Nivel educativo anterior aprobado
    "s04b_12"        # Categoría ocupacional
]

# Cargar archivos originales
df_discriminacion, meta_discriminacion = pyreadstat.read_sav(
    str(RAW_DIR / "EH2025_Discriminacion.sav"),
    usecols=variables_discriminacion
)

df_persona, meta_persona = pyreadstat.read_sav(
    str(RAW_DIR / "EH2025_Persona.sav"),
    usecols=variables_persona
)

# Crear conexión DuckDB
con = duckdb.connect()

# Registrar DataFrames
con.register("discriminacion", df_discriminacion)
con.register("persona", df_persona)

# Integrar ambas bases mediante las claves validadas
df_analitico = con.execute("""
    SELECT
        d.*,
        p.s01a_02,
        p.s01a_03,
        p.s01a_08,
        p.s01a_09,
        p.s01a_09npioc,
        p.s03a_03a,
        p.s04b_12
    FROM discriminacion AS d
    INNER JOIN persona AS p
        ON d.folio = p.folio
       AND d.nro = p.nro
""").df()

# Crear indicador general de discriminación
respuestas = df_analitico[variables_motivos]

tiene_discriminacion = respuestas.eq(1).any(axis=1)
todos_no = respuestas.eq(2).all(axis=1)

df_analitico["discriminacion_general"] = 2
df_analitico.loc[todos_no, "discriminacion_general"] = 0
df_analitico.loc[tiene_discriminacion, "discriminacion_general"] = 1

# Cantidad de motivos reportados
df_analitico["cantidad_motivos"] = respuestas.eq(1).sum(axis=1)

# Guardar base integrada
df_analitico.to_parquet(
    PROCESSED_DIR / "base_analitica_caso3.parquet",
    index=False
)

df_analitico.to_csv(
    PROCESSED_DIR / "base_analitica_caso3.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Base analítica del Caso 3 construida y guardada.")


In [ ]:
# Guardar base analítica en Parquet y CSV

df_analitico.to_parquet(
    PROCESSED_DIR / "base_analitica_caso3.parquet",
    index=False
)

df_analitico.to_csv(
    PROCESSED_DIR / "base_analitica_caso3.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Base analítica guardada correctamente.")

In [ ]:
%pip install ydata-profiling

In [ ]:
%pip install "setuptools<81"

In [ ]:
# Recuperar librerías y rutas
from pathlib import Path
import pandas as pd
from ydata_profiling import ProfileReport

current_dir = Path.cwd()
candidates = [current_dir, *current_dir.parents]

PROJECT_DIR = next(
    path for path in candidates
    if (path / "data" / "raw").is_dir()
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

# Cargar base analítica
df_analitico = pd.read_parquet(
    PROCESSED_DIR / "base_analitica_caso3.parquet"
)

# Generar reporte EDA
perfil = ProfileReport(
    df_analitico,
    title="Análisis Exploratorio - EH2025 Caso 3",
    explorative=True
)

# Guardar reporte
perfil.to_file(
    str(OUTPUTS_DIR / "reporte_eda_caso3.html")
)

print("Reporte EDA generado correctamente.")

In [ ]:
%pip install plotly folium scikit-learn xgboost shap

In [ ]:
# Preparar carpetas de visualizaciones
FIGURES_DIR = PROJECT_DIR / "outputs" / "figures"
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Entorno preparado para las visualizaciones.")

In [ ]:
import plotly.express as px

# Variables y nombres de los motivos
motivos_nombres = {
    "s09a_01a": "Color de piel",
    "s09a_01b": "Pertenencia indígena",
    "s09a_01c": "Procedencia o nacionalidad",
    "s09a_01d": "Orientación sexual o identidad de género",
    "s09a_01e": "Edad",
    "s09a_01f": "Sexo",
    "s09a_01g": "Idioma",
    "s09a_01h": "Vestimenta",
    "s09a_01i": "Discapacidad",
    "s09a_01j": "Religión",
    "s09a_01k": "Condición económica o social",
    "s09a_01l": "Otro motivo"
}

# Calcular respuestas afirmativas
resumen_motivos = pd.DataFrame({
    "motivo": motivos_nombres.values(),
    "personas": [
        (df_analitico[variable] == 1).sum()
        for variable in motivos_nombres
    ]
})

resumen_motivos = resumen_motivos.sort_values(
    "personas", ascending=True
)

# Crear gráfico interactivo
fig = px.bar(
    resumen_motivos,
    x="personas",
    y="motivo",
    orientation="h",
    text="personas",
    title="Principales motivos de discriminación en Bolivia - EH2025",
    labels={
        "personas": "Personas que reportaron el motivo",
        "motivo": "Motivo de discriminación"
    },
    template="plotly_white"
)

fig.update_traces(textposition="outside")
fig.update_layout(
    title_x=0.5,
    height=650,
    xaxis_title="Cantidad de personas",
    yaxis_title=""
)

# Guardar y mostrar
fig.write_html(
    str(FIGURES_DIR / "01_motivos_discriminacion.html")
)

fig.show()

In [ ]:
import plotly.express as px

# Nombres de los departamentos de Bolivia
departamentos = {
    1: "Chuquisaca",
    2: "La Paz",
    3: "Cochabamba",
    4: "Oruro",
    5: "Potosí",
    6: "Tarija",
    7: "Santa Cruz",
    8: "Beni",
    9: "Pando"
}

# Conservar respuestas determinadas
df_mapa = df_analitico[
    df_analitico["discriminacion_general"].isin([0, 1])
].copy()

df_mapa["departamento"] = (
    df_mapa["depto"].map(departamentos)
)

# Calcular tasa por departamento
resumen_departamentos = (
    df_mapa.groupby("departamento")
    .agg(
        personas=("discriminacion_general", "count"),
        casos=("discriminacion_general", "sum")
    )
    .reset_index()
)

resumen_departamentos["tasa_discriminacion"] = (
    resumen_departamentos["casos"] /
    resumen_departamentos["personas"] * 100
).round(2)

# Coordenadas geográficas aproximadas de capitales departamentales
coordenadas = {
    "Chuquisaca": (-19.0196, -65.2619),
    "La Paz": (-16.4897, -68.1193),
    "Cochabamba": (-17.3895, -66.1568),
    "Oruro": (-17.9647, -67.1060),
    "Potosí": (-19.5836, -65.7531),
    "Tarija": (-21.5355, -64.7296),
    "Santa Cruz": (-17.7833, -63.1821),
    "Beni": (-14.8333, -64.9000),
    "Pando": (-11.0267, -68.7692)
}

resumen_departamentos["latitud"] = resumen_departamentos[
    "departamento"
].map(lambda x: coordenadas[x][0])

resumen_departamentos["longitud"] = resumen_departamentos[
    "departamento"
].map(lambda x: coordenadas[x][1])

# Crear mapa interactivo
fig_mapa = px.scatter_map(
    resumen_departamentos,
    lat="latitud",
    lon="longitud",
    size="personas",
    color="tasa_discriminacion",
    hover_name="departamento",
    hover_data={
        "personas": True,
        "casos": True,
        "tasa_discriminacion": ":.2f",
        "latitud": False,
        "longitud": False
    },
    color_continuous_scale="Reds",
    zoom=4,
    height=650,
    title="Tasa de discriminación por departamento - EH2025"
)

fig_mapa.update_layout(
    map_style="open-street-map",
    title_x=0.5
)

# Guardar mapa
fig_mapa.write_html(
    str(FIGURES_DIR / "02_mapa_discriminacion_departamentos.html")
)

fig_mapa.show()

In [ ]:
import plotly.express as px
import pandas as pd

# Preparar datos
df_multivariado = df_analitico[
    df_analitico["discriminacion_general"].isin([0, 1])
].copy()

# Etiquetas de sexo
df_multivariado["sexo"] = df_multivariado["s01a_02"].map({
    1: "Hombre",
    2: "Mujer"
})

# Etiquetas de área
df_multivariado["area_geografica"] = df_multivariado["area"].map({
    1: "Urbana",
    2: "Rural"
})

# Agrupar edades
df_multivariado["grupo_edad"] = pd.cut(
    df_multivariado["s01a_03"],
    bins=[0, 17, 29, 44, 59, 120],
    labels=[
        "0-17 años",
        "18-29 años",
        "30-44 años",
        "45-59 años",
        "60 años o más"
    ]
)

# Calcular tasas de discriminación
resumen_multivariado = (
    df_multivariado.groupby(
        ["sexo", "grupo_edad", "area_geografica"],
        observed=True
    )
    .agg(
        personas=("discriminacion_general", "count"),
        casos=("discriminacion_general", "sum")
    )
    .reset_index()
)

resumen_multivariado["porcentaje"] = (
    resumen_multivariado["casos"] /
    resumen_multivariado["personas"] * 100
).round(2)

# Crear gráfico interactivo
fig_multi = px.bar(
    resumen_multivariado,
    x="grupo_edad",
    y="porcentaje",
    color="sexo",
    barmode="group",
    facet_col="area_geografica",
    hover_data=["personas", "casos"],
    title="Discriminación por sexo, edad y área geográfica - EH2025",
    labels={
        "grupo_edad": "Grupo de edad",
        "porcentaje": "Personas que reportaron discriminación (%)",
        "sexo": "Sexo",
        "area_geografica": "Área"
    },
    template="plotly_white"
)

fig_multi.update_layout(
    title_x=0.5,
    height=600
)

# Guardar y mostrar
fig_multi.write_html(
    str(FIGURES_DIR / "03_discriminacion_perfil_sociodemografico.html")
)

fig_multi.show()


In [ ]:

import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# ------------------------------------------------
# 1. Preparar variables sociodemográficas
# ------------------------------------------------

variables_categoricas = [
    "s01a_02",       # Sexo
    "s01a_08",       # Primer idioma aprendido
    "s01a_09",       # Autoidentificación indígena
    "s01a_09npioc",  # Nación o pueblo
    "s03a_03a",      # Nivel educativo
    "s04b_12",       # Categoría ocupacional
    "depto",         # Departamento
    "area"           # Área urbana/rural
]

variable_numerica = ["s01a_03"]  # Edad

variables_cluster = variables_categoricas + variable_numerica

df_cluster = df_analitico[
    variables_cluster + ["discriminacion_general"]
].copy()

# ------------------------------------------------
# 2. Preprocesamiento
# ------------------------------------------------

transformador = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            Pipeline([
                ("imputar", SimpleImputer(strategy="most_frequent")),
                ("codificar", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ))
            ]),
            variables_categoricas
        ),
        (
            "numericas",
            Pipeline([
                ("imputar", SimpleImputer(strategy="median")),
                ("escalar", StandardScaler())
            ]),
            variable_numerica
        )
    ]
)

X = transformador.fit_transform(df_cluster[variables_cluster])

# ------------------------------------------------
# 3. Aplicar K-Means
# ------------------------------------------------

modelo_kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

df_cluster["cluster"] = modelo_kmeans.fit_predict(X)

# ------------------------------------------------
# 4. Reducir dimensiones para visualizar
# ------------------------------------------------

pca = PCA(n_components=2, random_state=42)
coordenadas_pca = pca.fit_transform(X)

df_cluster["componente_1"] = coordenadas_pca[:, 0]
df_cluster["componente_2"] = coordenadas_pca[:, 1]

# ------------------------------------------------
# 5. Resumen de los clusters
# ------------------------------------------------

resumen_clusters = (
    df_cluster.groupby("cluster")
    .agg(
        personas=("cluster", "size"),
        edad_promedio=("s01a_03", "mean"),
        casos_discriminacion=(
            "discriminacion_general",
            lambda x: (x == 1).sum()
        ),
        respuestas_validas=(
            "discriminacion_general",
            lambda x: x.isin([0, 1]).sum()
        )
    )
    .reset_index()
)

resumen_clusters["tasa_discriminacion"] = (
    resumen_clusters["casos_discriminacion"] /
    resumen_clusters["respuestas_validas"] * 100
).round(2)

resumen_clusters["edad_promedio"] = (
    resumen_clusters["edad_promedio"].round(1)
)

display(resumen_clusters)

# ------------------------------------------------
# 6. Gráfico interactivo de clustering
# ------------------------------------------------

fig_cluster = px.scatter(
    df_cluster,
    x="componente_1",
    y="componente_2",
    color=df_cluster["cluster"].astype(str),
    hover_data=[
        "s01a_03",
        "s01a_02",
        "depto",
        "area",
        "discriminacion_general"
    ],
    title="Segmentación sociodemográfica mediante K-Means - EH2025",
    labels={
        "componente_1": "Componente principal 1",
        "componente_2": "Componente principal 2",
        "color": "Cluster"
    },
    opacity=0.65,
    template="plotly_white"
)

fig_cluster.update_layout(
    title_x=0.5,
    height=650
)

fig_cluster.write_html(
    str(FIGURES_DIR / "04_clustering_perfiles_sociodemograficos.html")
)

fig_cluster.show()

# ------------------------------------------------
# 7. Guardar resultados
# ------------------------------------------------

resumen_clusters.to_csv(
    TABLES_DIR / "resumen_clusters.csv",
    index=False,
    encoding="utf-8-sig"
)

df_cluster.to_csv(
    PROCESSED_DIR / "personas_clusters.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Clustering K-Means completado.")


In [ ]:

import numpy as np
import pandas as pd

# Convertir X a DataFrame y reemplazar pd.NA por np.nan
X = pd.DataFrame(X).copy()
X = X.astype(object)
X = X.mask(pd.isna(X), np.nan)

# Limpiar el objetivo y excluir únicamente filas sin clasificación
y = pd.Series(y).reset_index(drop=True)
X = X.reset_index(drop=True)

y = pd.to_numeric(y, errors="coerce")
filas_validas = y.notna()

X = X.loc[filas_validas].reset_index(drop=True)
y = y.loc[filas_validas].astype(int).to_numpy()

print("Datos preparados para validación cruzada.")


In [ ]:

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)

# ------------------------------------------------
# 1. Predicciones mediante validación cruzada
# ------------------------------------------------

prob_rf = cross_val_predict(
    modelo_rf, X, y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

prob_xgb = cross_val_predict(
    modelo_xgb, X, y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Clasificación con umbral de 0.5
pred_rf = (prob_rf >= 0.5).astype(int)
pred_xgb = (prob_xgb >= 0.5).astype(int)

# ------------------------------------------------
# 2. Calcular métricas
# ------------------------------------------------

resultados = pd.DataFrame([
    {
        "Modelo": "Random Forest",
        "AUC-ROC": roc_auc_score(y, prob_rf),
        "F1-Score": f1_score(y, pred_rf, zero_division=0),
        "Precision": precision_score(y, pred_rf, zero_division=0),
        "Recall": recall_score(y, pred_rf, zero_division=0)
    },
    {
        "Modelo": "XGBoost",
        "AUC-ROC": roc_auc_score(y, prob_xgb),
        "F1-Score": f1_score(y, pred_xgb, zero_division=0),
        "Precision": precision_score(y, pred_xgb, zero_division=0),
        "Recall": recall_score(y, pred_xgb, zero_division=0)
    }
])

display(resultados.round(4))

resultados.to_csv(
    TABLES_DIR / "metricas_modelos.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------
# 3. Matrices de confusión
# ------------------------------------------------

cm_rf = confusion_matrix(y, pred_rf, labels=[0, 1])
cm_xgb = confusion_matrix(y, pred_xgb, labels=[0, 1])

fig_cm = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Random Forest", "XGBoost"),
    horizontal_spacing=0.15
)

for matriz, nombre, columna in [
    (cm_rf, "Random Forest", 1),
    (cm_xgb, "XGBoost", 2)
]:
    fig_cm.add_trace(
        go.Heatmap(
            z=matriz,
            x=["Predicción: No", "Predicción: Sí"],
            y=["Real: No", "Real: Sí"],
            text=matriz,
            texttemplate="%{text}",
            colorscale="Blues",
            showscale=False,
            name=nombre
        ),
        row=1,
        col=columna
    )

fig_cm.update_layout(
    title="Matrices de confusión - EH2025",
    title_x=0.5,
    height=500,
    template="plotly_white"
)

fig_cm.write_html(
    str(FIGURES_DIR / "05_matrices_confusion.html")
)

fig_cm.show()

print("Evaluación de Random Forest y XGBoost completada.")


In [ ]:

import pandas as pd
import numpy as np
import plotly.express as px

# ------------------------------------------------
# 1. Extraer los modelos internos de los Pipeline
# ------------------------------------------------

rf_final = modelo_rf.steps[-1][1]
xgb_final = modelo_xgb.steps[-1][1]

importancia_rf = rf_final.feature_importances_
importancia_xgb = xgb_final.feature_importances_

# ------------------------------------------------
# 2. Obtener nombres de variables transformadas
# ------------------------------------------------

preprocesador_rf = modelo_rf[:-1]
nombres_variables = preprocesador_rf.get_feature_names_out()

# ------------------------------------------------
# 3. Construir tabla comparativa
# ------------------------------------------------

importancias = pd.DataFrame({
    "Variable": nombres_variables,
    "Random Forest": importancia_rf,
    "XGBoost": importancia_xgb
})

importancias["Promedio"] = importancias[
    ["Random Forest", "XGBoost"]
].mean(axis=1)

importancias = importancias.sort_values(
    "Promedio",
    ascending=False
)

display(importancias.head(20).round(5))

importancias.to_csv(
    TABLES_DIR / "importancia_variables_modelos.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------
# 4. Gráfico comparativo
# ------------------------------------------------

importancias_grafico = importancias.head(20).melt(
    id_vars="Variable",
    value_vars=["Random Forest", "XGBoost"],
    var_name="Modelo",
    value_name="Importancia"
)

fig_importancia = px.bar(
    importancias_grafico,
    x="Importancia",
    y="Variable",
    color="Modelo",
    barmode="group",
    orientation="h",
    title="Importancia de variables: Random Forest vs XGBoost",
    template="plotly_white"
)

fig_importancia.update_layout(
    title_x=0.5,
    height=750,
    yaxis={"categoryorder": "total ascending"}
)

fig_importancia.write_html(
    str(FIGURES_DIR / "06_importancia_variables.html")
)

fig_importancia.show()

print("Análisis XAI completado correctamente.")


In [ ]:

from pathlib import Path

DOCS_DIR = PROJECT_DIR / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta docs preparada.")



In [ ]:

import pandas as pd
import numpy as np
from datetime import date
from pathlib import Path
import plotly.io as pio
from IPython.display import display, HTML

# ------------------------------------------------
# 1. Recuperar resultados del análisis
# ------------------------------------------------

metricas = pd.read_csv(TABLES_DIR / "metricas_modelos.csv")
importancias = pd.read_csv(
    TABLES_DIR / "importancia_variables_modelos.csv"
)

# Resultados descriptivos
total_personas = len(df_analitico)

casos_discriminacion = (
    df_analitico["discriminacion_general"] == 1
).sum()

respuestas_validas = (
    df_analitico["discriminacion_general"].isin([0, 1])
).sum()

tasa_discriminacion = (
    casos_discriminacion / respuestas_validas * 100
)

# Motivos más reportados
motivos = {
    "s09a_01a": "Color de piel",
    "s09a_01b": "Pertenencia indígena",
    "s09a_01c": "Procedencia regional o nacionalidad",
    "s09a_01d": "Orientación sexual o identidad de género",
    "s09a_01e": "Edad",
    "s09a_01f": "Sexo",
    "s09a_01g": "Idioma",
    "s09a_01h": "Vestimenta",
    "s09a_01i": "Discapacidad",
    "s09a_01j": "Religión",
    "s09a_01k": "Condición económica o social",
    "s09a_01l": "Otro motivo"
}

resumen_motivos_final = pd.DataFrame([
    {
        "Motivo": nombre,
        "Casos": int((df_analitico[variable] == 1).sum())
    }
    for variable, nombre in motivos.items()
])

resumen_motivos_final = resumen_motivos_final.sort_values(
    "Casos", ascending=False
)

# Departamento con mayor tasa observada
resumen_deptos_final = (
    df_analitico[
        df_analitico["discriminacion_general"].isin([0, 1])
    ]
    .groupby("depto")
    .agg(
        casos=("discriminacion_general", "sum"),
        total=("discriminacion_general", "count")
    )
    .reset_index()
)

resumen_deptos_final["tasa"] = (
    resumen_deptos_final["casos"] /
    resumen_deptos_final["total"] * 100
)

# ------------------------------------------------
# 2. Preparar gráficos existentes
# ------------------------------------------------

archivos_graficos = [
    "01_motivos_discriminacion.html",
    "02_mapa_discriminacion_departamentos.html",
    "03_discriminacion_perfil_sociodemografico.html",
    "04_clustering_perfiles_sociodemograficos.html",
    "05_matrices_confusion.html",
    "06_importancia_variables.html"
]

graficos_html = []

for archivo in archivos_graficos:
    ruta = FIGURES_DIR / archivo

    if ruta.exists():
        contenido = ruta.read_text(encoding="utf-8")

        inicio = contenido.find("<div")
        fin = contenido.rfind("</div>") + 6

        if inicio >= 0 and fin > inicio:
            fragmento = contenido[inicio:fin]
            graficos_html.append(
                f"<section><h2>{archivo.replace('.html','').replace('_',' ').title()}</h2>"
                f"{fragmento}</section>"
            )

# ------------------------------------------------
# 3. Resultados de modelos
# ------------------------------------------------

tabla_metricas = metricas.round(4).to_html(
    index=False,
    classes="tabla",
    border=0
)

tabla_importancia = importancias.head(10).round(5).to_html(
    index=False,
    classes="tabla",
    border=0
)

tabla_motivos = resumen_motivos_final.head(5).to_html(
    index=False,
    classes="tabla",
    border=0
)

mejor_modelo = metricas.loc[
    metricas["AUC-ROC"].idxmax(), "Modelo"
]

mejor_auc = metricas["AUC-ROC"].max()

variables_destacadas = ", ".join(
    importancias.head(5)["Variable"].astype(str).tolist()
)

# ------------------------------------------------
# 4. Recomendaciones basadas en los resultados
# ------------------------------------------------

motivo_principal = resumen_motivos_final.iloc[0]["Motivo"]
motivo_principal_casos = int(
    resumen_motivos_final.iloc[0]["Casos"]
)

recomendaciones = f"""
<ol>
<li><b>Prevención focalizada:</b> priorizar campañas de prevención,
sensibilización y atención institucional relacionadas con
{motivo_principal}, el motivo con mayor número de respuestas afirmativas
en la base analizada ({motivo_principal_casos} casos).</li>

<li><b>Fortalecimiento de los mecanismos de denuncia:</b> mejorar la difusión,
accesibilidad y seguimiento de los canales de denuncia, considerando que
la base registra respuestas específicas sobre la presentación de denuncias.</li>

<li><b>Políticas territoriales de inclusión:</b> utilizar las diferencias
departamentales y los perfiles sociodemográficos observados para orientar
diagnósticos y acciones locales, sin interpretar asociaciones predictivas
como relaciones causales.</li>
</ol>
"""

# ------------------------------------------------
# 5. Construir documento HTML5
# ------------------------------------------------

html = f"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Informe Ejecutivo - EH2025 Caso 3</title>
<script src="https://cdn.plot.ly/plotly-3.0.1.min.js"></script>

<style>
body {{
    font-family: Arial, sans-serif;
    margin: 0;
    background: #f3f5f8;
    color: #263238;
    line-height: 1.6;
}}
header {{
    background: #17365d;
    color: white;
    padding: 45px 8%;
}}
main {{
    max-width: 1100px;
    margin: 30px auto;
    background: white;
    padding: 35px;
    border-radius: 10px;
}}
h1, h2 {{
    color: #17365d;
}}
header h1 {{
    color: white;
}}
section {{
    margin: 35px 0;
    padding-bottom: 20px;
    border-bottom: 1px solid #ddd;
}}
.tabla {{
    width: 100%;
    border-collapse: collapse;
    margin: 20px 0;
}}
.tabla th, .tabla td {{
    padding: 10px;
    border: 1px solid #ddd;
    text-align: left;
}}
.tabla th {{
    background: #e8eef5;
}}
.nota {{
    background: #f0f4f8;
    padding: 15px;
    border-left: 4px solid #17365d;
}}
footer {{
    text-align: center;
    padding: 25px;
    color: #666;
}}
</style>
</head>

<body>

<header>
<h1>Análisis Avanzado de Microdatos EH2025</h1>
<h2 style="color:white">Caso 3: Discriminación y Seguridad Ciudadana</h2>
<p>Informe ejecutivo de análisis exploratorio y Machine Learning</p>
<p>Fecha de elaboración: {date.today().strftime("%d/%m/%Y")}</p>
</header>

<main>

<section>
<h2>1. Resumen ejecutivo</h2>
<p>
El presente estudio analiza las respuestas sobre experiencias de
discriminación de la Encuesta de Hogares 2025 de Bolivia.
Se integraron variables de discriminación y características
sociodemográficas para estudiar patrones y construir modelos predictivos.
</p>
<p>
La base analítica contiene <b>{total_personas:,}</b> registros.
Se identificaron <b>{casos_discriminacion:,}</b> personas con al menos
un motivo de discriminación reportado entre
<b>{respuestas_validas:,}</b> respuestas clasificables.
La proporción observada en estos registros es
<b>{tasa_discriminacion:.2f}%</b>.
</p>
<div class="nota">
Este porcentaje es descriptivo y no constituye una estimación nacional
ponderada. No se aplicaron factores de expansión en este cálculo.
</div>
</section>

<section>
<h2>2. Objetivo y metodología</h2>
<p>
Identificar patrones sociodemográficos y geográficos asociados con la
discriminación reportada y evaluar la capacidad predictiva de dos
algoritmos: Random Forest y XGBoost.
</p>
<ul>
<li>Integración de microdatos mediante DuckDB.</li>
<li>Exploración automatizada con YData-Profiling.</li>
<li>Visualizaciones interactivas mediante Plotly.</li>
<li>Segmentación no supervisada mediante K-Means.</li>
<li>Clasificación supervisada con validación cruzada estratificada.</li>
<li>Interpretabilidad mediante importancia de variables.</li>
</ul>
</section>

<section>
<h2>3. Principales motivos reportados</h2>
<p>Los cinco motivos con más respuestas afirmativas fueron:</p>
{tabla_motivos}
</section>

<section>
<h2>4. Resultados de Machine Learning</h2>
<p>
Los modelos se evaluaron mediante predicciones fuera de muestra
obtenidas con validación cruzada estratificada.
</p>
{tabla_metricas}
<p>
El modelo con mayor AUC-ROC fue <b>{mejor_modelo}</b>,
con un valor de <b>{mejor_auc:.4f}</b>.
</p>
</section>

<section>
<h2>5. Interpretabilidad del modelo</h2>
<p>
Las variables con mayor importancia promedio entre ambos modelos fueron:
{variables_destacadas}.
</p>
{tabla_importancia}
<p>
La importancia de variables describe contribuciones predictivas
internas de los modelos; no demuestra causalidad.
</p>
</section>

<section>
<h2>6. Visualizaciones y resultados gráficos</h2>
{''.join(graficos_html)}
</section>

<section>
<h2>7. Recomendaciones de políticas públicas</h2>
{recomendaciones}
</section>

<section>
<h2>8. Limitaciones del estudio</h2>
<ul>
<li>Los porcentajes descriptivos presentados son no ponderados.</li>
<li>El mapa utiliza puntos representativos de capitales departamentales,
no límites geográficos municipales.</li>
<li>La segmentación K-Means identifica perfiles estadísticos,
no categorías sociales definitivas.</li>
<li>La base analítica utilizada no contiene una variable binaria
verificada de victimización delictiva. Por tanto, no se construyó
un modelo de victimización ni se presentan resultados de inseguridad
delictiva como si hubieran sido medidos.</li>
<li>Los modelos identifican asociaciones predictivas y no relaciones
causales.</li>
</ul>
</section>

<section>
<h2>9. Fuente de información</h2>
<p>
Fuente: Instituto Nacional de Estadística, Encuesta de Hogares 2025 -
<a href="http://anda.ine.gob.bo/index.php/catalog/256">
http://anda.ine.gob.bo/index.php/catalog/256</a>.
</p>
<p>
Fecha de acceso/descarga: registrar la fecha efectiva de descarga
de los microdatos originales.
</p>
<p>Uso académico de datos anonimizados.</p>
</section>

</main>

<footer>
Proyecto EH2025 - Caso 3 | Informe ejecutivo
</footer>

</body>
</html>
"""

ruta_informe = DOCS_DIR / "informe_ejecutivo_caso3.html"

ruta_informe.write_text(
    html,
    encoding="utf-8"
)

print("Informe ejecutivo generado:", ruta_informe)
